# Reproduce EPA's AERTEST in the browser

EPA tests every AERMOD release on a set of example problems and publishes the results.
This notebook runs the first of them, AERTEST (a stack with building downwash and four
days of Albany, NY meteorology), with the WebAssembly AERMOD and compares its 1-hour
concentrations with EPA's, receptor by receptor.

pyaermod's continuous integration runs the same comparison on every change to the
WebAssembly build and fails if any receptor differs by more than 0.01 %.

> **Not for regulatory use.** AERMOD here is EPA's public-domain source compiled to WebAssembly by pyaermod, not EPA's executable. Every change to it is checked against EPA's published AERTEST results (see the second notebook), but use EPA's executable for permit or compliance modeling.

In [ ]:
# Installs into this browser tab only; takes a few seconds the first time.
%pip install -q pyaermod aermod_wasm matplotlib

In [ ]:
import re
from pathlib import Path

# EPA's deck reads ../meteorology/ and writes ../plotfiles/ and ../Outputs/;
# here everything is in data/.
deck = Path("data/aertest.inp").read_text()
deck = re.sub(r"\.\./(meteorology|Outputs|plotfiles|postfiles)/", "", deck, flags=re.I)
deck = deck.replace("aermet2.sfc", "AERMET2.SFC").replace("aermet2.pfl", "AERMET2.PFL")
Path("data/aertest_here.inp").write_text(deck)
print(f"data/aertest_here.inp: {len(deck.splitlines())} lines")

In [ ]:
from pyaermod import AERMODRunner

result = AERMODRunner().run("data/aertest_here.inp")
print(result)

In [ ]:
import pandas as pd
from pyaermod import read_plotfile

def by_receptor(path):
    plot = read_plotfile(path)
    df = plot.to_dataframe()
    return df.set_index(["X", "Y"])[plot.concentration_column]

ours = by_receptor("data/AERTEST_01H.PLT")
epa = by_receptor("data/EPA_AERTEST_01H.PLT")
compare = pd.DataFrame({"browser": ours, "EPA": epa})
compare["relative difference"] = (compare["browser"] - compare["EPA"]).abs() / compare["EPA"].abs()
print(f"{len(compare)} receptors; largest relative difference {compare['relative difference'].max():.2e}")
compare.sort_values("EPA", ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(compare["EPA"], compare["browser"], s=12)
top = compare[["EPA", "browser"]].values.max() * 1.05
ax.plot([0, top], [0, top], color="grey", linewidth=1)
ax.set_xlabel("EPA's published 1-hour concentration (ug/m3)")
ax.set_ylabel("This browser's (ug/m3)")
ax.set_title("AERTEST, 144 receptors")
ax.set_aspect("equal")
plt.show()